# All of Us slide panels for the ASHG 2026 deck — standalone

The right-hand panel of each UK Biobank | All of Us pair (deck slides 26–29),
drawn to match the UKB panel beside it. **Standalone**: upload this one file
and run it. It needs numpy, pandas and matplotlib, the merged bin tables, and
nothing from the repo.

**Input:** `<WORK_DIR>/<phenotype>__<transform>__<covset>_merged.full.tsv` from
`08_accumulate.ipynb`, the bins file it was merged with, and — for the error
bars — the matching `_merged.jk.tsv`.

**Output** (`OUT_DIR`): one finished panel per trait — points, parent-offspring,
fits and the dashed line — as PNG (300 dpi) and PDF at 5.55 × 3.30 in (insert
at 100%), plus the plotted values under `tables/`. `wide/` is the slide
version; `pairs/` a matching pairs-per-bin panel (below); `wide_se/` is the
same curve panels, same axes, with
**±1 SE** delete-block jackknife bars on every bin mean (the convention 11
uses for bin means). The SE of a pooled display bin is recomputed from the
pooled replicates, not combined from its parts.

**Pairs per bin** (`pairs/<stem>.png|pdf`, counts in `tables/<stem>_pairs.csv`):
the same size and x axis as that trait's curve panel, so it can sit beneath
it; all pairs, full sibs and parent-offspring as step lines on a log axis. The
display bins differ in width (0.005 / 0.01 / 0.02), so the y axis is pairs
per 0.01 of `a_ij` — raw counts would jump at every change of width. Counts
under `MIN_PAIRS_PER_BIN` are neither drawn nor written out. They are pairs,
not people: each person is in many pairs. Counts are per trait, since only
people with the phenotype are in its pairs.

**Which traits:** the four slide traits by default. `ALL_TRAITS = True` (in
Settings) adds a panel for every trait merged at `TRANSFORM` / `COVSET`, for
the supplementary; those scale their own y axis, having no UKB panel to match.

What goes in:

- **Points**: `noPO` bin means (`other` + `FS`); **parent-offspring** bins as
  triangles, in no fit. Bins with fewer than `MIN_PAIRS_PER_BIN` pairs are not
  drawn. Points stop at 0.6, where the UKB panels stop. The merge's 0.001-wide
  bins below 0.02 are pooled to UKB's 0.005 for drawing (bin sums add, so a
  pooled mean is exact); the fits always use the merge's own bins.
- **Fits**, on every populated `noPO` bin in range, by pair-weighted least
  squares — the same fits `he_estimators` makes:

| legend | fitted on | design | `he_estimators` estimator |
|---|---|---|---|
| FS | 0.4–0.6 | slope + intercept | `b2_FS` |
| HS | 0.2–0.3 (UKB's HS window) | slope + intercept | none |
| Rel | 0.02–0.7 | slope + intercept | `h2_Rel` |
| OneSlopeOffsets | 0.09–0.7, bands at 0.18 / 0.36 | one slope, an intercept per band | `h2_OneSlopeOffsets` |

- **Dashed line**: `y = h2_Unrel · a_ij`, the slope from unrelated pairs
  (`a_ij < 0.02`, through the origin), labelled h²_SNP (Unrel).
- **y axis**: the paired UKB panel's range wherever every point and fit fits
  inside it; otherwise widened on that side only, and reported.

The look — size, Tenorite, margins, legend, line weights, colours — is copied
from `aou_slide_figures.py` in the deck's `UKB figures/` folder, which was
checked pixel-identical to the UKB panels. If that file changes, update the
style cell to match.

**Fonts:** put the Tenorite `.ttf` files in `FONT_DIR` (default `~/fonts`).
Without them the style cell stops: DejaVu Sans sets wider, so the panels would
not match the UKB ones and the error-bar legend would run off the panel. Set
`REQUIRE_TENORITE = False` to accept that anyway. Every legend is also checked
to fit its panel, and a panel whose legend does not fit stops the run.

**Leaving the Workbench:** figures and tables are aggregate and every drawn bin
has at least `MIN_PAIRS_PER_BIN` pairs, but they still go through the AoU
dissemination rules before they reach the deck.

## Settings

In [ ]:
import glob
import os
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.lines import Line2D
from IPython.display import Image, display

PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"
WORK_DIR    = os.path.expanduser(f"~/grm_pheno_cov_{RUN_DIR}")   # the merged tables
BINS_FILE   = f"{WORK_DIR}/bins_wide.txt"                         # the bins they were merged with
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
OUT_DIR  = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/{RUN_DIR}/07_estimators/slide_panels"
FONT_DIR = os.path.expanduser("~/fonts")                           # Tenorite .ttf files
PLOTS_MIRROR = os.path.expanduser(f"~/plots/{RUN_DIR}/slide_panels")

TRANSFORM, COVSET = "invnorm", "base_pcs"   # the reference combo, as in 11 and 11b
ALL_TRAITS = False   # True: also a panel for every other trait at TRANSFORM / COVSET
ERROR_BARS = True    # True: also write wide_se/, the same panels with +/-1 SE jackknife bars
REQUIRE_TENORITE = True   # stop if the deck font is missing; False allows DejaVu Sans (wider, won't match UKB)

MIN_PAIRS_PER_BIN = 50     # display only; every fit uses every populated bin
SLIDE_MAX_A = 0.6          # points and lines stop where the UKB panels' do
DISPLAY_BIN = 0.005        # UKB's bin width; narrower display bins are pooled up to it
PO_XOFF = 0.004            # PO and FS share bins; nudge so a triangle cannot hide a circle

# Model constants -- the values in src/he_estimators.py
U_HI, R_HI = 0.02, 0.7
FS_RANGE = (0.4, 0.6)
HS_WINDOW = (0.2, 0.3)     # UKB's HS* window; not he_estimators' HS_RANGE (0.2-0.4)
OFFSET_BANDS = {"band1": (0.09, 0.18), "band2": (0.18, 0.36), "band3": (0.36, R_HI)}
BAND_LO = min(lo for lo, _ in OFFSET_BANDS.values())
NOPO = ["other", "FS"]

assert os.path.isfile(BINS_FILE), f"missing {BINS_FILE} — see 08_accumulate.ipynb"
EDGES = np.loadtxt(BINS_FILE)          # (lo, hi) per bin, in bin_index order
MID = EDGES.mean(axis=1)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(MID)} bins | {TRANSFORM} / {COVSET}\nout: {OUT_DIR}")

## Slide style

Copied from `aou_slide_figures.py` (deck folder `UKB figures/`), with one
addition: `panel()` takes a fixed `ylim`. Unrel is dropped from the fits, as
on the UKB panels — here it is the dashed line instead.

In [ ]:
SIZE = (5.55, 3.30)                       # inches, one panel of a UKB | All of Us pair
MARGINS = dict(L=0.78, B=0.58, R=0.12, T=0.62)
POINT, GRID, ZERO = "#333333", "#e5e5e5", "#b0b0b0"
FIT_LW, FIT_ALPHA = 1.8, 0.5
H2_STYLE = dict(color="#1a1a1a", lw=1.5, ls=(0, (4, 2.5)), alpha=0.9)
# the UKB panels say "(LDSC)"; this line is the unrelated-pair slope
H2_LABEL = r"$h^2_{\mathrm{SNP}}$ (Unrel)"
MODEL_COLORS = {
    "FS": "#be9b6e", "HS": "#5f9ea0", "RelOffsets": "#759672", "Rel": "#ce7b79",
    "OneSlopeOffsets": "#b5716d", "OneSlope": "#8e7dae",
}
RC = {
    "font.size": 12, "axes.titlesize": 14, "axes.labelsize": 13,
    "xtick.labelsize": 11.5, "ytick.labelsize": 11.5, "legend.fontsize": 11,
    "axes.linewidth": 0.9, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.major.width": 0.9, "ytick.major.width": 0.9,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}

for _f in glob.glob(os.path.join(FONT_DIR, "**", "*.ttf"), recursive=True):
    fm.fontManager.addfont(_f)
FONT = "Tenorite" if "Tenorite" in {f.name for f in fm.fontManager.ttflist} else "DejaVu Sans"
plt.rcParams.update(RC)
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold"})
# A fallback font is not a cosmetic problem: DejaVu Sans sets ~10% wider, so
# the panels stop matching the UKB ones beside them and the error-bar legend
# runs off the panel. Hence a hard stop rather than a warning that scrolls past.
if FONT != "Tenorite" and REQUIRE_TENORITE:
    raise RuntimeError(
        f"Tenorite not found in {FONT_DIR}. Upload the four Tenorite .ttf files there "
        f"and restart the kernel, or set REQUIRE_TENORITE = False to accept DejaVu Sans.")
print(f"font: {FONT}" + ("" if FONT == "Tenorite" else "  -- NOT the deck font"))


def check_legend(fig, lg, out):
    """Stop if a legend runs off its panel -- it would be cropped on the slide."""
    bb = lg.get_window_extent(fig.canvas.get_renderer())
    over = max(bb.x1 - fig.bbox.x1, bb.y1 - fig.bbox.y1) / fig.dpi
    if over > 0:
        plt.close(fig)
        raise RuntimeError(f"{os.path.basename(out)}: the legend runs {over:.2f} in off the "
                           f"panel in {FONT}" + ("" if FONT == "Tenorite" else
                           " -- install Tenorite (see the style cell)"))


def row_major(items, ncol):
    nrow = -(-len(items) // ncol)
    grid = [items[i * ncol:(i + 1) * ncol] for i in range(nrow)]
    return [row[c] for c in range(ncol) for row in grid if c < len(row)]


def panel(points, fits, variant, out, h2=None, ylim=None, se=False):
    w, h = SIZE
    m = MARGINS
    fig = plt.figure(figsize=SIZE)
    ax = fig.add_axes([m["L"] / w, m["B"] / h, (w - m["L"] - m["R"]) / w, (h - m["B"] - m["T"]) / h])
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.axhline(0, color=ZERO, lw=1.0, zorder=1)

    binned = points[points["group"] != "parent_offspring"]
    po = points[points["group"] == "parent_offspring"]
    ax.scatter(binned.x, binned.y, s=22, c=POINT, lw=0, zorder=4)
    if len(po):
        ax.scatter(po.x, po.y, s=30, c=POINT, marker="^", lw=0, zorder=4)

    unknown = set(fits.model) - set(MODEL_COLORS)
    if unknown:
        raise ValueError(f"no colour defined for model(s) {sorted(unknown)}")
    # every variant gets the with-fits axis ranges so build steps line up
    ax.update_datalim(list(zip(fits.x, fits.y)))
    ax.autoscale_view()
    if ylim is not None:
        ax.set_ylim(*ylim)
    if se:
        # freeze the limits first, so the whiskers cannot rescale the axes and
        # the wide_se/ panel keeps exactly the wide/ panel's; a whisker that
        # runs past the axis is clipped there
        ax.set_xlim(*ax.get_xlim()); ax.set_ylim(*ax.get_ylim())
        for d in (binned, po):
            ok = np.isfinite(d["se"].to_numpy())
            if ok.any():
                ax.errorbar(d["x"][ok], d["y"][ok], yerr=d["se"][ok], fmt="none", ecolor=POINT,
                            elinewidth=0.8, alpha=0.45, capsize=0, zorder=3.9)
    if variant == "with_fits":
        for (model, _), seg in fits.groupby(["model", "segment"], sort=False):
            ax.plot(seg.x, seg.y, color=MODEL_COLORS[model], lw=FIT_LW, alpha=FIT_ALPHA, zorder=3,
                    solid_capstyle="butt")
    has_h2 = variant != "points_only" and h2 is not None
    if has_h2:
        x0, x1 = ax.get_xlim()
        ax.plot([x0, x1], [h2 * x0, h2 * x1], zorder=3.5, scalex=False, scaley=False, **H2_STYLE)

    ax.locator_params(axis="y", nbins=5)
    ax.set_xlabel(r"Genetic relatedness ($a_{ij}$)")
    ax.set_ylabel(r"Mean cross-product ($\tilde{y}_i\,\tilde{y}_j$)")

    hs = [Line2D([], [], ls="none", marker="o", ms=5, mfc=POINT, mec="none")]
    labs = ["Binned data (±1 SE)" if se else "Binned data"]
    if len(po):
        hs.append(Line2D([], [], ls="none", marker="^", ms=5.5, mfc=POINT, mec="none"))
        labs.append("Parent–offspring")
    if variant == "with_fits":
        for model in MODEL_COLORS:  # fixed legend order
            if model in set(fits.model):
                hs.append(Line2D([], [], color=MODEL_COLORS[model], lw=2.4)); labs.append(model)
    if has_h2:
        hs.append(Line2D([], [], **H2_STYLE)); labs.append(H2_LABEL)
    lg = ax.legend(row_major(hs, 4), row_major(labs, 4), loc="lower left",
                   bbox_to_anchor=(0.03, 1 - (0.62 - 0.1) / 3.30 + 0.02), bbox_transform=fig.transFigure,
                   ncol=4, frameon=False, handlelength=1.5, handletextpad=0.45, columnspacing=0.9,
                   labelspacing=0.3, borderaxespad=0, borderpad=0)
    check_legend(fig, lg, out)
    lims = (ax.get_xlim(), ax.get_ylim())
    fig.savefig(out + ".png", dpi=300)
    fig.savefig(out + ".pdf")
    plt.close(fig)
    return lims


# Pairs per bin. Class hues are covplot's CLASS_COL for FS and PO (checked for
# colourblind separation); all pairs in the points' ink.
PAIR_STYLE = {"all": (POINT, "All pairs"), "FS": ("#eb6834", "Full sibs"),
              "PO": ("#1baf7a", "Parent–offspring")}
PER_A = 0.01                    # y unit: pairs per this much a_ij


def pairs_panel(counts, xlim, out):
    """Pairs per display bin, per class, as step lines on a log axis, on the
    curve panel's x axis. `counts`: lo, hi, n_all, n_FS, n_PO (NaN = not drawn)."""
    w, h = SIZE
    m = MARGINS
    fig = plt.figure(figsize=SIZE)
    ax = fig.add_axes([m["L"] / w, m["B"] / h, (w - m["L"] - m["R"]) / w, (h - m["B"] - m["T"]) / h])
    ax.grid(True, color=GRID, lw=0.8, zorder=0)
    ax.set_axisbelow(True)
    lo, hi = counts["lo"].to_numpy(), counts["hi"].to_numpy()
    hs, labs = [], []
    for key, (col, lab) in PAIR_STYLE.items():
        dens = counts[f"n_{key}"].to_numpy(float) / (hi - lo) * PER_A
        ok = np.flatnonzero(np.isfinite(dens))
        if not len(ok):
            continue
        # one step line per run of drawn bins; a blanked bin leaves a gap
        for run in np.split(ok, np.flatnonzero(np.diff(ok) > 1) + 1):
            ax.plot(np.column_stack([lo[run], hi[run]]).ravel(), np.repeat(dens[run], 2),
                    color=col, lw=1.6, zorder=3, solid_joinstyle="miter")
        hs.append(Line2D([], [], color=col, lw=2.4)); labs.append(lab)
    ax.set_yscale("log")
    ax.set_xlim(*xlim)
    ax.set_xlabel(r"Genetic relatedness ($a_{ij}$)")
    ax.set_ylabel(r"Pairs per 0.01 of $a_{ij}$")
    lg = ax.legend(row_major(hs, 4), row_major(labs, 4), loc="lower left",
                   bbox_to_anchor=(0.03, 1 - (0.62 - 0.1) / 3.30 + 0.02), bbox_transform=fig.transFigure,
                   ncol=4, frameon=False, handlelength=1.5, handletextpad=0.45, columnspacing=0.9,
                   labelspacing=0.3, borderaxespad=0, borderpad=0)
    check_legend(fig, lg, out)
    fig.savefig(out + ".png", dpi=300)
    fig.savefig(out + ".pdf")
    plt.close(fig)

## Slides

One row per All of Us panel. `ukb_ylim` is the paired UKB panel's y range,
recomputed from `ukb/publication/extracted_fits.json` with the autoscale
`make_slide_figures.py` gives a wide panel (points plus the FS, HS, Rel and
OneSlopeOffsets fits, 5% margin). Set it to `None` to let a panel scale itself.
Alcohol pairs AUDIT-C here with UKB's *alcohol intake frequency*.

In [ ]:
SLIDES = pd.DataFrame([
    # phenotype,              file stem,             slide, ukb_ylim                 # paired UKB trait
    ("waist_circumference",   "Waist_circumference", 26, (-0.088687, 0.521584)),    # Waist circumference
    ("bmi",                   "Body_mass_index_BMI", 27, (-0.055967, 0.439216)),    # Body mass index (BMI)
    ("height",                "Standing_height",     28, (-0.040250, 0.689520)),    # Standing height
    ("alcohol_audit_c_score", "Alcohol_use_AUDIT_C", 29, (-0.034370, 0.354878)),    # Alcohol intake frequency
], columns=["phenotype", "stem", "slide", "ukb_ylim"])

if ALL_TRAITS:
    # every other trait merged at this combo, for the supplementary: named by
    # phenotype, and scaling its own y axis (no UKB panel to match)
    _suffix = f"__{TRANSFORM}__{COVSET}_merged.full.tsv"
    _found = sorted(os.path.basename(p)[: -len(_suffix)] for p in glob.glob(f"{WORK_DIR}/*{_suffix}"))
    _extra = [p for p in _found if p not in set(SLIDES["phenotype"])]
    SLIDES = pd.concat([SLIDES, pd.DataFrame({"phenotype": _extra, "stem": _extra,
                                              "slide": "supp", "ukb_ylim": None})],
                       ignore_index=True)

SLIDES["full_tsv"] = [f"{WORK_DIR}/{p}__{TRANSFORM}__{COVSET}_merged.full.tsv"
                      for p in SLIDES["phenotype"]]
SLIDES["jk_tsv"] = SLIDES["full_tsv"].str.replace(".full.tsv", ".jk.tsv", regex=False)
missing = SLIDES.loc[~SLIDES["full_tsv"].map(os.path.isfile), "phenotype"].tolist()
assert not missing, f"no {TRANSFORM} / {COVSET} merge in {WORK_DIR} for: {', '.join(missing)}"
if ERROR_BARS:
    no_jk = SLIDES.loc[~SLIDES["jk_tsv"].map(os.path.isfile), "phenotype"].tolist()
    assert not no_jk, f"ERROR_BARS needs the .jk.tsv too; missing for: {', '.join(no_jk)}"
print(f"{len(SLIDES)} traits ({(SLIDES['slide'] != 'supp').sum()} on slides)")
print(SLIDES[["slide", "phenotype", "stem", "ukb_ylim"]].to_string(index=False))

## Bins and fits

`load_sums` reads the full-data sums and pair counts per class and bin.
Each fit is pair-weighted least squares on bin means over every populated
`noPO` bin in its range — identical to pair-level OLS with each pair at its
bin midpoint, and to what `he_estimators` fits. No fit crosses `a_ij = 0.02`.

In [ ]:
CLASSES = ["other", "FS", "PO"]


def load_sums(path):
    """{class: (sum, n)} arrays over the bin grid, from a *_merged.full.tsv."""
    f = pd.read_csv(path, sep="\t", usecols=["class", "bin_index", "full_sum", "full_n"])
    out = {}
    for c in CLASSES:
        s, n = np.zeros(len(MID)), np.zeros(len(MID))
        fc = f[f["class"] == c]
        k = fc["bin_index"].to_numpy() - 1
        s[k], n[k] = fc["full_sum"].to_numpy(), fc["full_n"].to_numpy()
        out[c] = (s, n)
    return out


def load_jk(path):
    """{class: (sum, n)} arrays of shape (blocks, bins), from a *_merged.jk.tsv.
    Row b holds the sums with jackknife block b left out."""
    j = pd.read_csv(path, sep="\t", usecols=["class", "block", "bin_index", "jk_sum", "jk_n"])
    b0, b1, nb = int(j["block"].min()), int(j["block"].max()), j["block"].nunique()
    assert b0 == 0 and nb == b1 + 1, f"{path}: blocks run {b0}-{b1} with {nb} present"
    out = {}
    for c in CLASSES:
        s, n = np.zeros((nb, len(MID))), np.zeros((nb, len(MID)))
        jc = j[j["class"] == c]
        b, k = jc["block"].to_numpy(), jc["bin_index"].to_numpy() - 1
        s[b, k], n[b, k] = jc["jk_sum"].to_numpy(), jc["jk_n"].to_numpy()
        out[c] = (s, n)
    return out


def class_sums(sums, classes):
    return (sum(sums[c][0] for c in classes), sum(sums[c][1] for c in classes))


def band_ind(a):
    return np.column_stack([((a >= lo) & (a < hi)).astype(float) for lo, hi in OFFSET_BANDS.values()])


_origin = lambda a: a[:, None]
_slope_int = lambda a: np.c_[a, np.ones_like(a)]
# legend name -> (fitted ranges, design)
FITS = {
    "FS":              ([FS_RANGE],       _slope_int),
    "HS":              ([HS_WINDOW],      _slope_int),
    "Rel":             ([(U_HI, R_HI)],   _slope_int),
    "OneSlopeOffsets": ([(BAND_LO, R_HI)], lambda a: np.c_[a, band_ind(a)]),
}
UNREL = ([(-np.inf, U_HI)], _origin)    # the dashed line's slope

for _name, (_ranges, _) in {**FITS, "Unrel": UNREL}.items():
    _sides = {"u" if hi <= U_HI else "r" if lo >= U_HI else None for lo, hi in _ranges}
    assert None not in _sides and len(_sides) == 1, f"{_name} spans a_ij = {U_HI}"


def wls(X, m, n):
    """Pair-weighted least squares; None if under-determined."""
    if len(m) <= X.shape[1]:
        return None
    w = np.sqrt(n)
    Xw = X * w[:, None]
    if np.linalg.matrix_rank(Xw) < X.shape[1]:
        return None
    return np.linalg.lstsq(Xw, m * w, rcond=None)[0]


def fit(sums, ranges, design):
    """(beta, a of the bins fitted) on noPO, every populated bin in range."""
    s, n = class_sums(sums, NOPO)
    inside = np.zeros(len(MID), dtype=bool)
    for lo, hi in ranges:
        inside |= (MID >= lo) & (MID < hi)
    k = inside & (n > 0)
    a = MID[k]
    return (wls(design(a), s[k] / n[k], n[k]) if k.any() else None), a


def _display_grid():
    """Display cells: bins narrower than DISPLAY_BIN pooled into DISPLAY_BIN
    cells, wider bins kept. Returns (cell midpoints, bins x cells 0/1 matrix)."""
    lo, hi = np.rint(EDGES[:, 0] * 1000).astype(int), np.rint(EDGES[:, 1] * 1000).astype(int)
    d = int(round(DISPLAY_BIN * 1000))
    keys = [((l // d) * d, (l // d) * d + d) if h - l < d else (l, h) for l, h in zip(lo, hi)]
    cells = sorted(set(keys))
    col = {k: i for i, k in enumerate(cells)}
    P = np.zeros((len(keys), len(cells)))
    P[np.arange(len(keys)), [col[k] for k in keys]] = 1.0
    # a pooled cell must be tiled exactly, or its mean covers a range it does not say
    covered = (hi - lo) @ P
    bad = [c for c, w in zip(cells, covered) if w != c[1] - c[0]]
    assert not bad, f"bins do not tile the {DISPLAY_BIN} display cells at {bad[:3]}"
    return (np.array([(a + b) / 2000 for a, b in cells]), P,
            np.array([a / 1000 for a, _ in cells]), np.array([b / 1000 for _, b in cells]))


CELL_MID, POOL, CELL_LO, CELL_HI = _display_grid()


def pair_counts(sums):
    """Pairs per display cell up to SLIDE_MAX_A -- all classes, FS, PO. A count
    under MIN_PAIRS_PER_BIN is blanked (NaN): not drawn, not written out."""
    keep = CELL_MID <= SLIDE_MAX_A
    out = pd.DataFrame({"lo": CELL_LO[keep], "hi": CELL_HI[keep]})
    for key, classes in (("all", CLASSES), ("FS", ["FS"]), ("PO", ["PO"])):
        n = (class_sums(sums, classes)[1] @ POOL)[keep]
        out[f"n_{key}"] = np.where(n >= MIN_PAIRS_PER_BIN, n, np.nan)
    return out


def display_bins(sums, classes, jks=None):
    """(mid, mean, n, se) on the display grid. Sums add, so a pooled mean is the
    exact pair-weighted mean. `se` is the delete-block jackknife SE of that
    mean, from the pooled replicates -- NaN without `jks`, or where a replicate
    leaves the cell empty."""
    s, n = class_sums(sums, classes)
    S, N = s @ POOL, n @ POOL
    k = N > 0
    mean = np.divide(S, N, out=np.full(len(N), np.nan), where=k)
    se = np.full(len(N), np.nan)
    if jks is not None:
        sr, nr = class_sums(jks, classes)
        Sr, Nr = sr @ POOL, nr @ POOL                       # (blocks, cells)
        B = len(Sr)
        full = (Nr > 0).all(axis=0)
        reps = np.divide(Sr, Nr, out=np.zeros_like(Sr), where=Nr > 0)
        dev = reps - reps.mean(axis=0)
        se = np.where(full, np.sqrt((B - 1) / B * (dev ** 2).sum(axis=0)), np.nan)
    return CELL_MID[k], mean[k], N[k], se[k]


BAND_EDGES = sorted({e for r in OFFSET_BANDS.values() for e in r})


def segments(lo, hi):
    """[lo, hi] split at every band edge inside it, stopping just short of each
    cut so a step function is never evaluated in the next band."""
    cuts = [lo] + [e for e in BAND_EDGES if lo < e < hi] + [hi]
    return [(k0, k1 - 1e-9 if k1 < hi else k1)
            for k0, k1 in zip(cuts[:-1], cuts[1:]) if k1 - k0 > 1e-9]


def slide_data(path, jk_path=None):
    """points (with SE if jk_path), fits (polylines), h2_Unrel, and the
    coefficients, for one trait."""
    sums = load_sums(path)
    jks = load_jk(jk_path) if jk_path else None
    rows = []
    for classes, group, xoff in ((NOPO, "binned", 0.0), (["PO"], "parent_offspring", PO_XOFF)):
        a, m, n, se = display_bins(sums, classes, jks)
        keep = (a <= SLIDE_MAX_A) & (n >= MIN_PAIRS_PER_BIN)
        rows.append(pd.DataFrame({"x": a[keep] + xoff, "y": m[keep], "se": se[keep],
                                  "n_pairs": n[keep].astype(int), "group": group}))
    points = pd.concat(rows, ignore_index=True)
    binned = points[points["group"] == "binned"]
    assert len(binned), f"{path}: no bin with >= {MIN_PAIRS_PER_BIN} pairs"
    x_lo, x_hi = binned["x"].min(), binned["x"].max()

    fits, coefs, seg = [], {}, 0
    for name, (ranges, design) in FITS.items():
        beta, a_fit = fit(sums, ranges, design)
        coefs[name] = beta
        if beta is None or not len(a_fit):
            continue
        lo, hi = max(a_fit.min(), x_lo), min(a_fit.max(), x_hi)
        for k0, k1 in (segments(lo, hi) if hi > lo else []):
            xs = np.linspace(k0, k1, 60)
            fits.append(pd.DataFrame({"model": name, "segment": seg, "x": xs, "y": design(xs) @ beta}))
            seg += 1
    beta_u, _ = fit(sums, *UNREL)
    h2u = float(beta_u[0]) if beta_u is not None else np.nan
    return points, pd.concat(fits, ignore_index=True), h2u, coefs


print(f"fits: {', '.join(FITS)}; dashed line: Unrel slope")

## Cross-check against `he_estimators` (optional)

If the repo is on this machine, the fits above are checked against what
`he_estimators` reports for the same merge. Without it this cell says so and
moves on — the notebook does not depend on it.

In [ ]:
import sys

_repo = next((p for p in (os.environ.get("AOUCOV_REPO", ""),
                          os.path.expanduser("~/repos/aou-covariance"),
                          os.path.expanduser("~/aou-covariance"))
              if p and os.path.isfile(os.path.join(p, "src", "he_estimators.py"))), None)
if _repo is None:
    print("he_estimators not found -- cross-check skipped")
else:
    sys.path.insert(0, os.path.join(_repo, "src"))
    import he_estimators as HE
    for r in SLIDES.itertuples():
        jk = r.full_tsv.replace(".full.tsv", ".jk.tsv")
        S, N = HE.load_arrays(r.full_tsv, jk, len(MID), HE.blocks_in(jk))
        est = HE.estimates(S, N, 0, MID)
        _, _, h2u, coefs = slide_data(r.full_tsv)
        # a display cell that is one merge bin must have he_estimators' bin SE
        if ERROR_BARS:
            a_he, m_he, n_he, se_he = HE.bin_curve(S, N, HE.NOPO, MID, HE.blocks_in(jk))
            a_d, m_d, n_d, se_d = display_bins(load_sums(r.full_tsv), NOPO, load_jk(jk))
            one = POOL.sum(axis=0) == 1
            single = set(np.round(CELL_MID[one], 6))
            k_he = np.array([round(x, 6) in single for x in a_he])
            k_d = np.array([round(x, 6) in single for x in a_d])
            assert np.allclose(m_d[k_d], m_he[k_he]) and np.allclose(
                se_d[k_d], se_he[k_he], equal_nan=True), f"{r.phenotype}: bin SEs disagree"
        read = lambda name, f: f(coefs[name]) if coefs[name] is not None else np.nan
        for key, ours in (("h2_Unrel", h2u),
                          ("b2_FS", read("FS", lambda b: 2 * b[1])),
                          ("h2_Rel", read("Rel", lambda b: b[0])),
                          ("h2_OneSlopeOffsets", read("OneSlopeOffsets", lambda b: b[0]))):
            theirs = est[(key, "noPO")]
            assert (np.isnan(ours) and np.isnan(theirs)) or np.isclose(
                ours, theirs, rtol=1e-9, atol=1e-12), (
                f"{r.phenotype}: {key} = {ours:.6g} here, {theirs:.6g} in he_estimators")
    print(f"matches he_estimators ({_repo}) for all {len(SLIDES)} traits")

## Render

One panel per trait: `wide/<stem>.png|pdf`, its error-bar twin in
`wide_se/<stem>.png|pdf`, the pairs-per-bin panel in `pairs/<stem>.png|pdf`,
and the plotted values under `tables/`.

In [ ]:
SETS = ["wide"] + (["wide_se"] if ERROR_BARS else [])
for d in SETS + ["pairs"]:
    os.makedirs(f"{OUT_DIR}/{d}", exist_ok=True)
os.makedirs(f"{OUT_DIR}/tables", exist_ok=True)


def slide_ylim(points, fits, ukb):
    """The UKB range, widened only on a side an All of Us point or fit would
    leave, with a 5% margin there. Returns (ylim, matched); None -> autoscale."""
    if ukb is None:
        return None, False
    ys = np.r_[points["y"].to_numpy(), fits["y"].to_numpy(), 0.0]
    lo, hi = float(ys.min()), float(ys.max())
    pad = 0.05 * (hi - lo)
    y0, y1 = ukb
    ylim = (y0 if lo >= y0 else lo - pad, y1 if hi <= y1 else hi + pad)
    return ylim, ylim == tuple(ukb)


params = []
for r in SLIDES.itertuples():
    points, fits, h2u, _ = slide_data(r.full_tsv, r.jk_tsv if ERROR_BARS else None)
    ylim, matched = slide_ylim(points, fits, r.ukb_ylim)
    points.to_csv(f"{OUT_DIR}/tables/{r.stem}_points.csv", index=False)
    fits.to_csv(f"{OUT_DIR}/tables/{r.stem}_fits.csv", index=False)
    params.append({"slide": r.slide, "phenotype": r.phenotype, "h2_unrel": h2u,
                   "ylim_lo": ylim[0] if ylim else np.nan,
                   "ylim_hi": ylim[1] if ylim else np.nan, "y_matches_ukb": matched})
    h2_line = h2u if np.isfinite(h2u) else None     # no line rather than a NaN line with a key
    # the same ylim for both sets, so a wide_se/ panel can replace its wide/ twin
    (xlim, _) = panel(points, fits, "with_fits", f"{OUT_DIR}/wide/{r.stem}", h2_line, ylim)
    if ERROR_BARS:
        panel(points, fits, "with_fits", f"{OUT_DIR}/wide_se/{r.stem}", h2_line, ylim, se=True)
    # pairs per bin, on the curve panel's own x axis
    counts = pair_counts(load_sums(r.full_tsv))
    counts.to_csv(f"{OUT_DIR}/tables/{r.stem}_pairs.csv", index=False)
    pairs_panel(counts, xlim, f"{OUT_DIR}/pairs/{r.stem}")
    n_po = int((points["group"] == "parent_offspring").sum())
    yl = ("y = UKB" if matched else "own y" if ylim is None
          else f"y widened to ({ylim[0]:.2f}, {ylim[1]:.2f}), UKB ({r.ukb_ylim[0]:.2f}, {r.ukb_ylim[1]:.2f})")
    print(f"slide {r.slide}  {r.phenotype:<22} {len(points) - n_po:>3} bins, {n_po} PO, "
          f"smallest {points['n_pairs'].min():,} pairs | h2_Unrel {h2u:.3f} | {yl}")
    print(f"           pairs drawn: all {np.nansum(counts['n_all']):,.0f}, "
          f"FS {np.nansum(counts['n_FS']):,.0f}, PO {np.nansum(counts['n_PO']):,.0f}")
    if r.slide != "supp":       # inline: the slide traits only
        for d in SETS + ["pairs"]:
            display(Image(filename=f"{OUT_DIR}/{d}/{r.stem}.png", width=555))

pd.DataFrame(params).to_csv(f"{OUT_DIR}/tables/panel_params.csv", index=False)
print(f"\n-> {OUT_DIR}/{{{','.join(SETS)}}}/")

## Copy to the home directory and zip

Everything — `wide/`, `wide_se/`, `pairs/` and `tables/` — into `PLOTS_MIRROR` under the
home directory, then one zip of it there (`~/slide_panels_<run>.zip`). The
zip is what to download, **after** the AoU dissemination check.

In [ ]:
for _d in SETS + ["pairs", "tables"]:
    shutil.copytree(f"{OUT_DIR}/{_d}", f"{PLOTS_MIRROR}/{_d}", dirs_exist_ok=True)
ZIP = shutil.make_archive(os.path.expanduser(f"~/slide_panels_{RUN_DIR}"), "zip", PLOTS_MIRROR)
_n = {d: len(glob.glob(f"{PLOTS_MIRROR}/{d}/*.png")) for d in SETS + ["pairs"]}
print(f"{PLOTS_MIRROR}: " + ", ".join(f"{d}/ {k} panels" for d, k in _n.items()) + ", tables/")
print(f"zip: {ZIP}  ({os.path.getsize(ZIP) / 1e6:.1f} MB)")